# MEER Script-to-Video — Kaggle (hamesha free)

Script likho → video bane. Na library chahiye, na credit, na paise.

- **Free GPU:** Kaggle T4 x2 (har hafte 30 GPU-ghante free, phir reset).
- **Model:** LTX-Video 2B distilled — har line ka apna 5-second clip, asal motion.
- **Kaam:** apni script cell 1 me likho, Run All dabao, har clip ~5-6 minute. Aakhir me `meer-final.mp4` (video + awaz + subtitles) Output se download karo.
- **Resume:** agar session mar jaye to dobara Run All dabao — ban chuke clips dobara nahi bante (checkpoint `worker-state.json`).


In [ ]:
# ---- EDIT THIS CELL: apni script yahan likho ----
# Har line = aik scene (5 second). Pehli dafa test ke liye 3 lines rakhi hain.
# Zyada lines chahiye to neeche MAX_CLIPS bhi barha do.
SCRIPT = """Rayo drives fast along a sunny country road, passing green fields and small houses.
Rayo sees a small puppy waiting near a wooden bridge and slows down.
Rayo stops safely and asks the puppy what happened."""

HERO = "Rayo"
CHARACTER_DESCRIPTION = ("a sleek red 3D cartoon race car with a yellow lightning bolt mark, "
                         "windshield eyes and a friendly bumper smile")
STYLE = "polished 3D animated children's movie, bright colors, soft cinematic lighting, family-friendly"

# MOTION RULE (proven 2026-10-26, run 18399d2c): front-approach — car drives
# TOWARD the camera, front three-quarter view, grows larger as it approaches,
# camera COMPLETELY still. Side view wali video me chehra kho jata hai aur gari
# frame se bahar nikal jati hai — ye nahi.
MOTION = ("drives toward the camera along the road, front three-quarter view, "
          "the car grows larger as it approaches, background and road moving past, "
          "wheels turning, camera completely still, the car never leaves the frame")

# PER-LINE MOTION (quality tests 2026-10-26, runs A2 + COMBO): when
# FRONT-APPROACH (proven 2026-10-26, run 18399d2c): driving/together lines
# now drive TOWARD the camera (front three-quarter, grows larger as it
# approaches) — 9/9 face samples clear, car in frame 9/9. Lateral drive
# lost the face and drove out of frame in 4 of 5 earlier runs. Stop rule
# unchanged (its control run was fine).
# PER_LINE_MOTION is True, each script line picks its own motion sentence
# by keyword instead of the single MOTION above. A2 proved per-line motion
# makes clips differ per scene (camel appears late in camel-lines); the
# framing guard ("keeps it centered in frame, never leaves the frame") is
# the fix for A2 clip 1 / C v3 clips 1+3, where the car drove out of frame.
# Default OFF — current behavior unchanged.
PER_LINE_MOTION = False
# Rule order matters: "together" first (test-script line 3 contains both
# "helps"/"together" AND "race" — it must map to the together-motion, not
# the fast-motion), then stop/find, then fast/race.
MOTION_RULES = [
    (("help", "helps", "together"),
     "races toward the camera together with the other character, front three-quarter view, "
     "joyful energetic motion, wheels turning, camera stays steady and keeps them centered "
     "in frame, they never leave the frame"),
    (("stop", "stops", "find", "finds", "see", "sees", "meet", "meets",
      "slow", "slows", "wait", "waits"),
     "slows to a gentle stop, the scene stays still around the car, subtle "
     "gentle motion, camera stays steady and keeps the car centered in frame, "
     "the car never leaves the frame"),
    (("race", "races", "zoom", "zooms", "fast", "chase", "speed", "speeds"),
     "drives toward the camera, front three-quarter view, the car grows larger as it "
     "approaches, dust rising behind the wheels, wheels turning, camera stays steady and "
     "keeps the car centered in frame, the car never leaves the frame"),
]
DEFAULT_MOTION = ("drives toward the camera along the road, front three-quarter view, the car "
                  "grows larger as it approaches, wheels turning, camera stays steady and keeps "
                  "the car centered in frame, the car never leaves the frame")

def motion_for_line(line):
    low = line.lower()
    for keywords, motion in MOTION_RULES:
        if any(k in low for k in keywords):
            return motion
    return DEFAULT_MOTION

MAX_CLIPS = 3          # pehli test ke liye 3; lambi kahani ke liye barha do (12 = ~1 min video)
SEED = 42
STORY_NAME = "meer-story"   # clips isi naam se GitHub par jayenge (agar token ho)

REPO = "Hamzaagsm/videomaker"
BRANCH = "main"

SCRIPT_LINES = [line.strip() for line in SCRIPT.splitlines() if line.strip()]
assert SCRIPT_LINES, "Add at least one non-empty script line."
MAX_CLIPS = min(MAX_CLIPS, len(SCRIPT_LINES))

QUEUE = []
for clip_no, line in enumerate(SCRIPT_LINES, 1):
    motion = motion_for_line(line) if PER_LINE_MOTION else MOTION
    motion_prompt = (
        f"{HERO}, {CHARACTER_DESCRIPTION}. Visual action for this scene: {line}. "
        f"{motion}. {STYLE}. Keep the same proportions, colors, eyes, mouth and "
        f"identifying marks. No redesign, no extra wheels or faces, clean composition, vertical 9:16."
    )
    QUEUE.append((clip_no, line, motion_prompt))

# Reference still: default Rayo DESERT v2 (blue eyes, smaller framing — switched 2026-10-26 after v2 test: face 7/9, setting 9/9 desert).
# Apni photo chahiye to Kaggle me "my-character.png" naam se upload karo —
# wo ho to ye cell usi ko use karega, warna default Rayo.
STILL_URL = ("https://raw.githubusercontent.com/Hamzaagsm/videomaker/main/"
             "test/colab-factory/rayo-desert-v2.png")
STILL_FILE = "character-reference.png"
OWN_STILL = "my-character.png"

def clip_repo_path(n):
    return f"test/colab-factory/meer-script-{STORY_NAME}-clip-{n}.mp4"

# LTX generation config (2B distilled: 8 steps, no CFG — guidance 1.0)
WIDTH, HEIGHT = 512, 768   # portrait 9:16, divisible by 32
NUM_FRAMES = 121           # PROVEN on T4 (Colab + Kaggle 2026-10-25/26).
                           # 193 OOMs (T4 sirf 14.56 GiB); 121 frames @ 24 fps = ~5 s per clip.
NUM_STEPS = 8              # distilled checkpoint: 8 steps is the whole point
                           # run.py --num-steps overrides (24 tested 2026-10-26:
                           # fixes the face smear, ~1.6x GPU-h)
GUIDANCE = 1.0             # distilled: no CFG/STG needed

# Official 2B distilled checkpoint (VERIFIED exists, 6,340,744,492 bytes):
# https://huggingface.co/Lightricks/LTX-Video (file ltxv-2b-0.9.8-distilled.safetensors)
LTX_2B_DISTILLED_URL = ("https://huggingface.co/Lightricks/LTX-Video/resolve/main/"
                        "ltxv-2b-0.9.8-distilled.safetensors")
LTX_FALLBACK_REPO = "multimodalart/ltxv-2b-0.9.6-distilled"

print("Config OK:", HERO, "| script lines:", len(SCRIPT_LINES),
      "| clips to make:", MAX_CLIPS, "| clip size:", WIDTH, "x", HEIGHT, "@24fps,", NUM_FRAMES, "frames")


In [ ]:
# ---- GPU CHECK: Kaggle par T4 x2 milna chahiye ----
# Right panel me Accelerator -> GPU T4 x2 ON aur Internet ON zaroori hain.
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
# ---- INSTALL: diffusers (LTX pipeline) + helpers ----
# PINNED versions — unpinned --upgrade ne pehle env bigara tha.
# Ye wohi versions hain jo 2026-10-25/26 ko OBSERVED working thin.
# Do NOT re-add --upgrade. Do NOT touch torch.
!pip install -q diffusers==0.42.0 transformers==5.20.2 accelerate==1.15.0 imageio-ffmpeg requests gTTS
print("install done")


## 1. Load LTX-Video 2B distilled (image-to-video)

**Load order (FIXED 2026-10-25 after proof run 1 — the as-written cell failed twice):**
1. **Weights first, local file:** download the OFFICIAL 0.9.8 2B distilled weights to disk (`ltxv-2b-0.9.8-distilled.safetensors`, 6,340,744,492 bytes — size-checked). NEVER pass the full `resolve/...` URL to `from_single_file` — diffusers prepends its own `resolve/main/` and 404s.
2. **No text encoder in the checkpoint:** the single file holds transformer + VAE only (944 keys, no T5). Load with `text_encoder=None, tokenizer=None` and pre-encode the prompts ONCE on CPU (T5-XXL from `PixArt-alpha/PixArt-XL-2-1024-MS`, `text_encoder/` + `tokenizer/` subfolders, bf16, `device_map="auto"` with GPU 1 GB / CPU 11 GB caps + disk offload) into `prompt_embeds.pt`; the worker passes `prompt_embeds` + `prompt_attention_mask` instead of `prompt=`.
3. **If the load still fails:** the cell prints the exact OFFICIAL-CLI fallback commands (clone `Lightricks/LTX-Video`, `pip install -e .[inference]`, `inference.py ...`) — do not guess, run those.

**VRAM survival on a 16 GB T4:** `enable_sequential_cpu_offload()` + `vae.enable_tiling()` + `vae.use_framewise_decoding = True` are MANDATORY.


In [ ]:
import gc, os, time, torch, urllib.request
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

# ---- FIX 1: download the weights to a LOCAL file first (size-checked) ----
# Passing the full https://huggingface.co/.../resolve/main/... URL to
# from_single_file FAILS: diffusers prepends its own resolve/main/ and 404s.
LTX_LOCAL = "ltxv-2b-0.9.8-distilled.safetensors"
LTX_SIZE = 6340744492  # bytes, verified on Hugging Face
t0 = time.perf_counter()
if not (os.path.exists(LTX_LOCAL) and os.path.getsize(LTX_LOCAL) == LTX_SIZE):
    print(f"Downloading LTX 2B distilled weights ({LTX_SIZE/1e9:.2f} GB)...")
    urllib.request.urlretrieve(LTX_2B_DISTILLED_URL, LTX_LOCAL)
assert os.path.getsize(LTX_LOCAL) == LTX_SIZE, "weights size mismatch — re-download"
print(f"Weights on disk: {LTX_LOCAL} ({os.path.getsize(LTX_LOCAL)} bytes, MATCH)")

# ---- FIX 2: the single-file checkpoint has NO text encoder (transformer + VAE only) ----
# Load the pipeline WITHOUT it, and pre-encode prompts separately (below).
pipe = None
load_path = None
try:
    pipe = LTXImageToVideoPipeline.from_single_file(
        LTX_LOCAL, torch_dtype=torch.bfloat16,
        text_encoder=None, tokenizer=None)
    load_path = "official 0.9.8 2B distilled (local file, text_encoder=None)"
except Exception as e:
    print("PRIMARY load failed:", type(e).__name__, str(e)[:300])
    print("OFFICIAL-CLI FALLBACK — run these cells instead:")
    print("  !git clone https://github.com/Lightricks/LTX-Video.git")
    print("  !pip install -q -e 'LTX-Video[inference]'")
    print("  !python LTX-Video/inference.py --prompt '<MOTION PROMPT>' --conditioning_media_paths character-reference.png --conditioning_start_frames 0 --height 768 --width 512 --num_frames 121 --seed 42 --pipeline_config LTX-Video/configs/ltxv-2b-0.9.8-distilled.yaml")
    raise RuntimeError("from_single_file failed — use the official CLI fallback above")

# MANDATORY on 16 GB T4: sequential CPU offload + VAE tiling + framewise decode.
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.use_framewise_decoding = True  # FIX 3: VAE decode OOMs (12.76 GiB) without this
t_load = time.perf_counter() - t0
print(f"Pipeline loaded OK via {load_path} in {t_load:.1f}s ({t_load/60:.1f} min)")

# ---- FIX 2 (continued): pre-encode ALL queue prompts ONCE on CPU ----
# Mirrors diffusers LTX _get_t5_prompt_embeds exactly (max_length 128, padding
# max_length, truncation, bf16). T5-XXL is ~19-22 GB — Kaggle's ~13 GB RAM
# cannot hold it, so it runs device_map="auto" (GPU 1 GB cap, CPU 11 GB cap,
# rest disk-offloaded). One-time cost; the encoder is freed before generation.
from transformers import T5EncoderModel, T5TokenizerFast
t_enc0 = time.perf_counter()
_tok = T5TokenizerFast.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS", subfolder="tokenizer")
# FIX 2b (2026-10-25 05:23): device_map="disk" DIED under transformers 5.20.2
# (ValueError — "disk" is no longer a valid device_map string). The 5.x way:
# device_map="auto" + max_memory caps + offload_folder. GPU capped at 1 GB so
# generation keeps its VRAM; CPU RAM capped at 11 GB; the rest spills to disk.
# Encoding prompts pages layers from disk — slow one-time cost, then freed.
_enc = T5EncoderModel.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS", subfolder="text_encoder",
    torch_dtype=torch.bfloat16, device_map="auto",
    max_memory={0: "1GB", "cpu": "11GB"}, offload_folder="t5-offload")
PROMPT_EMBEDS = {}
for clip_no, line, motion_prompt in QUEUE[:MAX_CLIPS]:
    _inp = _tok(motion_prompt, padding="max_length", max_length=128,
                truncation=True, add_special_tokens=True, return_tensors="pt")
    with torch.no_grad():
        _emb = _enc(_inp.input_ids)[0].to(torch.bfloat16)
    PROMPT_EMBEDS[clip_no] = (_emb, _inp.attention_mask.bool())
    print(f"  encoded clip {clip_no}: embeds {_emb.shape}")
torch.save(PROMPT_EMBEDS, "prompt_embeds.pt")
del _enc, _tok
gc.collect()
print(f"Prompts pre-encoded in {time.perf_counter()-t_enc0:.1f}s -> prompt_embeds.pt "
      f"({len(PROMPT_EMBEDS)} clips). T5 encoder freed.")


## 2. Reference still — the still pins the character

The still pins the character: image-to-video keeps Rayo = Rayo across clips (text-to-video would re-invent him every clip). Default still is Rayo SIDE VIEW from GitHub. Apni photo chahiye to Kaggle me `my-character.png` naam se upload karo — wo ho to wahi use hogi.


In [ ]:
import os, shutil, urllib.request
from PIL import Image

if os.path.exists(OWN_STILL):
    print("Using YOUR uploaded still:", OWN_STILL)
    shutil.copyfile(OWN_STILL, STILL_FILE)
elif not os.path.exists(STILL_FILE):
    print("Downloading default Rayo still from:", STILL_URL)
    try:
        urllib.request.urlretrieve(STILL_URL, STILL_FILE)
    except Exception as e:
        print("Download failed:", e)
        print("KAGGLE FALLBACK: upload your character photo as 'my-character.png' —")
        print("right panel > Add data > Upload (or Files > Upload), then re-run this cell.")
        raise RuntimeError("character still missing — upload my-character.png and re-run this cell")
assert os.path.exists(STILL_FILE), "character still missing — upload my-character.png and re-run this cell"
still = Image.open(STILL_FILE).convert("RGB")
print("Still:", STILL_FILE, still.size, os.path.getsize(STILL_FILE), "bytes")
# LTX input must match the generation size exactly:
still_small = still.resize((WIDTH, HEIGHT), Image.LANCZOS)
still_small.save("still-512x768.png")
print("Resized still for LTX input: still-512x768.png", still_small.size)


## 3. QUEUE WORKER LOOP — one clip per script line, checkpoint after each

For every script line: **skip if already done** (checkpoint file `worker-state.json` + staged file on disk, so re-running after a dead session resumes instead of redoing work) → generate the clip (**timed**) → stage with ffmpeg to the MEER contract (**timed**) → ffprobe-verify → **upload to GitHub immediately if a GITHUB_TOKEN Kaggle Secret exists** (**timed**) → checkpoint. One clip failing never kills the queue — it is marked `failed` and the worker moves on.


In [ ]:
import base64, json, os, subprocess, time, urllib.request, urllib.error

# ---- Prompt embeds: from the load cell (same session) or prompt_embeds.pt (re-run) ----
try:
    PROMPT_EMBEDS
except NameError:
    PROMPT_EMBEDS = torch.load("prompt_embeds.pt", map_location="cpu", weights_only=True)
    print(f"Loaded prompt_embeds.pt ({len(PROMPT_EMBEDS)} clips)")

# ---- GitHub token: Kaggle Secret (env var). Never printed. ----
# Add a Kaggle Secret named GITHUB_TOKEN (Add-ons > Secrets) to enable upload;
# without it, staged clips stay on disk — download them from Output.
# Upload is OPTIONAL: without a token the worker still generates + stages +
# ffprobe-verifies every clip and keeps the staged file on disk.
TOKEN = os.environ.get("GITHUB_TOKEN")
UPLOAD_ENABLED = bool(TOKEN)
if not UPLOAD_ENABLED:
    print("NOTE: GITHUB_TOKEN secret missing — uploads SKIPPED; staged clips stay on disk")
    print("      (download clip-*-staged.mp4 from Output)")

def _gh_headers():
    return {"Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {TOKEN}",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "meer-script-kaggle/1.0"}

def gh_exists(repo_path):
    if not TOKEN:
        return None
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/contents/{repo_path}?ref={BRANCH}",
        headers=_gh_headers(), method="GET")
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read().decode()).get("sha")
    except urllib.error.HTTPError as e:
        assert e.code == 404, f"GET {repo_path} failed: {e.code}"
        return None

def gh_upload(repo_path, local_path, message):
    with open(local_path, "rb") as f:
        content_b64 = base64.b64encode(f.read()).decode()
    base = f"https://api.github.com/repos/{REPO}/contents/{repo_path}"
    sha = gh_exists(repo_path)
    payload = {"message": message, "content": content_b64, "branch": BRANCH}
    if sha:
        payload["sha"] = sha
    req = urllib.request.Request(base, data=json.dumps(payload).encode(),
                                 headers={**_gh_headers(), "Content-Type": "application/json"},
                                 method="PUT")
    with urllib.request.urlopen(req, timeout=300) as r:
        data = json.loads(r.read().decode())
    assert r.status in (200, 201), f"PUT {repo_path} failed: {r.status}"
    print(f"  Uploaded {repo_path} -> sha {data['content']['sha'][:10]} size {data['content']['size']}")
    return data["content"]["sha"]

# ---- Checkpoint: survives a re-run in the same session; GitHub is the real checkpoint. ----
STATE_FILE = "worker-state.json"
state = {}
if os.path.exists(STATE_FILE):
    state = json.load(open(STATE_FILE))
    print("Resumed checkpoint:", {k: v.get("status") for k, v in state.items()})

def save_state():
    json.dump(state, open(STATE_FILE, "w"), indent=2)

def stage_clip(raw_path, staged_path):
    t0 = time.perf_counter()
    subprocess.run(["ffmpeg", "-y", "-i", raw_path,
                    "-vf", "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,fps=24",
                    "-an", "-c:v", "libx264", "-bf", "0", "-g", "48", "-crf", "22",
                    staged_path],
                   check=True, capture_output=True, text=True)
    return time.perf_counter() - t0

def ffprobe_check(path):
    out = subprocess.run(["ffprobe", "-v", "quiet", "-print_format", "json",
                          "-show_streams", "-show_format", path],
                         check=True, capture_output=True, text=True).stdout
    d = json.loads(out)
    s = [x for x in d["streams"] if x["codec_type"] == "video"][0]
    ok = (int(s["width"]) == 1080 and int(s["height"]) == 1920
          and s["r_frame_rate"].startswith("24") and int(s.get("has_b_frames", 1)) == 0
          and int(d["format"]["size"]) > 100000)
    print(f"  ffprobe: {s['width']}x{s['height']} fps {s['r_frame_rate']} "
          f"has_b_frames={s.get('has_b_frames')} duration={d['format']['duration']}s "
          f"size={d['format']['size']} -> {'PASS' if ok else 'FAIL'}")
    return ok

# ---- THE WORKER LOOP ----
results = []
for clip_no, line, motion_prompt in QUEUE[:MAX_CLIPS]:
    key = str(clip_no)
    repo_path = clip_repo_path(clip_no)
    # Resume: skip only if the checkpoint says done AND the staged clip is
    # really there (on disk, or on GitHub when a token exists).
    already = state.get(key, {}).get("status") == "done" and (
        os.path.exists(f"clip-{clip_no}-staged.mp4")
        or (UPLOAD_ENABLED and gh_exists(repo_path)))
    if already:
        print(f"[clip {clip_no}] SKIP — already done ({repo_path})")
        results.append(state[key])
        continue
    print(f"[clip {clip_no}] LINE: {line}")
    rec = {"clip_no": clip_no, "line": line, "status": "failed"}
    try:
        t_gen0 = time.perf_counter()
        _embeds, _mask = PROMPT_EMBEDS[clip_no]
        frames = pipe(image=still_small,
                      prompt_embeds=_embeds.to("cuda"),
                      prompt_attention_mask=_mask.to("cuda"),
                      width=WIDTH, height=HEIGHT,
                      num_frames=NUM_FRAMES,
                      num_inference_steps=NUM_STEPS,
                      guidance_scale=GUIDANCE,
                      generator=torch.Generator(device="cuda").manual_seed(SEED + clip_no),
                      ).frames[0]
        raw_path = f"clip-{clip_no}-raw.mp4"
        export_to_video(frames, raw_path, fps=24)
        rec["gen_s"] = round(time.perf_counter() - t_gen0, 1)
        print(f"  Generated {raw_path} in {rec['gen_s']:.1f}s ({rec['gen_s']/60:.1f} min), "
              f"{len(frames)} frames @24fps")

        staged_path = f"clip-{clip_no}-staged.mp4"
        rec["stage_s"] = round(stage_clip(raw_path, staged_path), 1)
        print(f"  Staged {staged_path} in {rec['stage_s']:.1f}s")
        rec["ffprobe_ok"] = ffprobe_check(staged_path)

        if UPLOAD_ENABLED:
            t_up0 = time.perf_counter()
            rec["sha"] = gh_upload(repo_path, staged_path,
                                   f"MEER script-to-video: {STORY_NAME} clip {clip_no} ({HERO})")
            rec["upload_s"] = round(time.perf_counter() - t_up0, 1)
        else:
            rec["upload_s"] = 0
            print(f"  (upload skipped — staged file kept on disk: {staged_path})")
        rec["total_s"] = round(rec["gen_s"] + rec["stage_s"] + rec["upload_s"], 1)
        rec["status"] = "done" if rec["ffprobe_ok"] else "done-unverified"
        print(f"  [clip {clip_no}] DONE in {rec['total_s']:.1f}s "
              f"({rec['total_s']/60:.1f} min) — status {rec['status']}")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {str(e)[:300]}"
        print(f"  [clip {clip_no}] FAILED: {rec['error']} — moving to next line")
    state[key] = rec
    results.append(rec)
    save_state()  # checkpoint after EVERY clip — a dead session loses at most one clip

print("QUEUE DONE —", sum(1 for r in results if r.get("status", "").startswith("done")),
      "/", len(results), "clips done")


In [ ]:
# ---- SUMMARY: measured minutes-per-clip -> videos per week (honest math) ----
import json

done = [r for r in results if r.get("status", "").startswith("done")]
print("=== MEER SCRIPT-TO-VIDEO — MEASURED SUMMARY ===")
print(f"Model load: {t_load:.1f}s ({t_load/60:.1f} min)  [MEASURED]")
for r in done:
    print(f"  clip {r['clip_no']}: gen {r.get('gen_s', 0):.0f}s + stage {r.get('stage_s', 0):.0f}s "
          f"+ upload {r.get('upload_s', 0):.0f}s = {r.get('total_s', 0):.0f}s "
          f"({r.get('total_s', 0)/60:.1f} min)  ffprobe={'PASS' if r.get('ffprobe_ok') else 'FAIL'}")
if done:
    avg_min = sum(r["total_s"] for r in done) / len(done) / 60
    print(f"\nAVERAGE per clip (MEASURED): {avg_min:.1f} min")
    print(f"One 24-line video needs ~{24 * avg_min / 60:.1f} GPU-hours.")
    print(f"Kaggle free = 30 GPU-hours/week -> ~{30 / (24 * avg_min / 60):.0f} videos of 24 lines per week.")
    print("That is the permanent free lane: every week the 30 hours come back.")
else:
    print("NO clips finished — the run failed. Report the error above; do not invent timings.")
print("Checkpoint file: worker-state.json (re-run resumes finished clips).")


In [ ]:
# ---- Build final MP4 + free narration + subtitle file (KAGGLE version) ----
# Outputs stay in /kaggle/working: download them from the Output panel
# (meer-final.mp4, meer-narration.mp3, meer-subtitles.srt).
# No auto-download here — on Kaggle you download from the Output panel.
import os, subprocess
from gtts import gTTS

done = [(n, line) for n, line, _ in QUEUE[:MAX_CLIPS]
        if os.path.exists(f"clip-{n}-staged.mp4")]
if not done:
    raise RuntimeError("No staged scenes found. Review generation errors above.")
done_clips = [f"clip-{n}-staged.mp4" for n, _ in done]
with open("meer-concat.txt", "w", encoding="utf-8") as f:
    for p in done_clips:
        f.write("file '" + os.path.abspath(p) + "'\n")
subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "meer-concat.txt",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-an", "meer-visuals.mp4"],
               check=True, capture_output=True, text=True)

# gTTS uses an external free web service, not a paid API; availability can change.
TTS_LANG = "en"  # change to "hi" for Hindi
gTTS(text="\n".join(line for _, line in done), lang=TTS_LANG, slow=False).save("meer-narration.mp3")
audio_probe = subprocess.run(["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
                              "-of", "default=noprint_wrappers=1:nokey=1", "meer-narration.mp3"],
                             check=True, capture_output=True, text=True).stdout.strip()
audio_seconds = float(audio_probe)
subprocess.run(["ffmpeg", "-y", "-stream_loop", "-1", "-i", "meer-visuals.mp4",
                "-i", "meer-narration.mp3", "-t", str(audio_seconds),
                "-map", "0:v:0", "-map", "1:a:0", "-c:v", "libx264", "-crf", "22",
                "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k", "-shortest",
                "meer-final.mp4"], check=True, capture_output=True, text=True)

def srt_time(seconds):
    ms = int(round(seconds * 1000))
    h, rem = divmod(ms, 3600000); m, rem = divmod(rem, 60000); s, ms = divmod(rem, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"
with open("meer-subtitles.srt", "w", encoding="utf-8") as f:
    for i, (_, line) in enumerate(done, 1):
        f.write(f"{i}\n{srt_time((i-1)*5)} --> {srt_time(i*5)}\n{line}\n\n")
print("FINAL OUTPUTS (in /kaggle/working — download from the Output panel):")
for p in ["meer-final.mp4", "meer-narration.mp3", "meer-subtitles.srt"]:
    print(p, os.path.getsize(p), "bytes")
print("SRT timing is approximate; review before publishing.")


## Limits (imaandari se)

- **Kaggle free: har hafte 30 GPU-ghante.** Khatam ho jayen to agle hafte reset — koi card, koi paise nahi. Aik 24-line video ~2-2.5 GPU-ghante leti hai, matlab hafte me ~12-15 videos.
- **GPU busy ho to:** "No GPU available" ya session fail ho to thodi der ruk kar dobara try karo (subah aam tor par khali milta hai). Ya Colab se chala lo — wahi notebook Colab par bhi chalta hai (free GPU kabhi kabhi busy hota hai).
- **Duniya me koi bhi free cheez unlimited nahi hoti** — lekin ye har hafte wapas milta hai, hamesha. Library sirf cache hai; ye rasta asal factory hai.
- **gTTS awaz:** free web service hai; kabhi fail ho to dobara chalao.
- **SRT timing approximate hai** — har line 5 second mani gayi hai.
- **Character:** aik hi reference photo se hero aik jaisa rehta hai; har line ka background prompt me alag likho to scene bhi badlega.
- **Kling/Google Flow jaisi web tools:** Rayo (Lightning McQueen jaisi) ko copyright filter rok deta hai — is liye ye LTX rasta hi asal hai.
- **REAL MOTION only:** stills/slideshows se video nahi banti — har clip asal me chalta hai.
